# Preprocessing Komentar YouTube Bahasa Indonesia dengan Spark + MongoDB

Notebook mandiri ini membaca data **hanya dari MongoDB** melalui MongoDB Spark Connector, memulai preprocessing dari `text_original`, dan menyimpan hasil **hanya ke MongoDB**.

Output teks final hanya satu: `text_final`. Kolom teks turunan lama dibuang lalu dibangun ulang dari `text_original`.

## Konfigurasi

Isi `.env` berdasarkan `.env.example`. Notebook tidak membaca CSV, JSON, dataset lokal, atau pandas.

In [1]:
from dotenv import load_dotenv
load_dotenv(dotenv_path=".env", override=True)
print('Konfigurasi .env dimuat. Pipeline data hanya membaca dan menulis MongoDB.')


Konfigurasi .env dimuat. Pipeline data hanya membaca dan menulis MongoDB.


## Implementasi Pipeline

Pipeline ini memperketat cleaning berdasarkan pembacaan penuh data preprocessed dan sample labeling: normalisasi typo/slang, penghapusan URL/mention, deteksi spam, normalisasi mixed alphanumeric, perlindungan negasi dan istilah domain, whitelist stemming untuk kata penting, stopword aman, serta output `text_final` yang lebih readable untuk labeling/training. Stemming default dimatikan agar kata seperti `pemerintahan`, `jabatan`, dan `melakukan` tidak berubah menjadi potongan kata yang sulit dibaca. Output MongoDB tetap fixed ke `comments_preprocessed`; report fixed ke `comments_preprocessing_report`.


In [2]:
from __future__ import annotations

import html
import json
import os
import re
import sys
import unicodedata
import uuid
from datetime import datetime, timezone
from typing import Any

import emoji
import ftfy
import regex
from dotenv import load_dotenv
from pymongo import MongoClient
from pyspark import StorageLevel
from pyspark.sql import DataFrame, SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    BooleanType,
    DoubleType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
)
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory


MONGO_CONNECTOR_PACKAGE = "org.mongodb.spark:mongo-spark-connector_2.13:11.0.1"
FINAL_TEXT_COLUMN = "text_final"
FIXED_OUTPUT_COLLECTION = "comments_preprocessed"
FIXED_REPORT_COLLECTION = "comments_preprocessing_report"
MAX_PREPROCESS_CHARS = 1000
MAX_PREPROCESS_TOKENS = 120
MIN_TRAIN_TOKENS = 3
DERIVED_TEXT_COLUMNS = {
    "text_clean",
    "text_preprocessed",
    "text_stemmed",
    "text_final",
    "text_final_classic",
    "text_final_transformer",
    "text_final_classic_stemmed",
    "text_stopword_removed",
    "text_slang_normalized",
}
OUTPUT_INPUT_COLUMNS = [
    "comment_id",
    "video_id",
    "label",
    "text_original",
]
GENERATED_COLUMNS = DERIVED_TEXT_COLUMNS | {
    "tokens",
    "normalized_token_count",
    "raw_char_count",
    "raw_word_count",
    "emoji_count",
    "url_count",
    "mention_count",
    "hashtag_count",
    "exclamation_count",
    "question_count",
    "uppercase_ratio",
    "is_all_caps",
    "contains_negation",
    "contains_domain_terms",
    "contains_profanity",
    "domain_term_count",
    "profanity_count",
    "is_duplicate_text",
    "duplicate_text_rank",
    "processed_at",
}

NEGATIONS = {"tidak", "bukan", "jangan", "belum", "tanpa", "kurang", "tak"}
INTENSIFIERS = {
    "sangat",
    "amat",
    "banget",
    "terlalu",
    "paling",
    "lebih",
    "semakin",
    "sekali",
}
DOMAIN_TERMS = {
    "tni",
    "dpr",
    "ruu",
    "uu",
    "sipil",
    "militer",
    "rakyat",
    "negara",
    "korupsi",
    "koruptor",
    "aset",
    "perampasan",
    "pemerintah",
    "presiden",
    "prabowo",
    "polisi",
    "demo",
    "mahasiswa",
    "orde",
    "orba",
    "oligarki",
    "demokrasi",
    "pasal",
}
SENTIMENT_TERMS = {
    "setuju",
    "mantap",
    "bagus",
    "buruk",
    "kacau",
    "rusak",
    "takut",
    "bahaya",
    "aman",
    "bravo",
    "bubarkan",
    "dukung",
    "tolak",
    "sahkan",
    "lawan",
    "adil",
    "zalim",
    "parah",
    "hancur",
    "ngeri",
    "bejat",
    "bacot",
    "boikot",
}
PROFANITY_TERMS = {
    "anjing",
    "anjir",
    "anjay",
    "bangsat",
    "bajingan",
    "brengsek",
    "goblok",
    "tolol",
    "bodoh",
    "kampret",
    "kontol",
    "memek",
    "ngentot",
    "tai",
    "sialan",
    "berak",
    "bacot",
}
SPECIAL_TOKENS = {
    "emo_laugh",
    "emo_sad",
    "emo_angry",
    "emo_pos",
    "emo_neg",
    "emo_think",
    "emo_other",
    "url_token",
    "user_mention",
    "ruu_tni",
    "uu_tni",
    "ruu_polri",
    "uu_polri",
    "ruu_perampasan_aset",
    "uu_perampasan_aset",
    "uud_1945",
    "uud_nri_1945",
    "angka_58_persen",
    "paslon_02",
}
STEM_WHITELIST = {
    "tni",
    "dpr",
    "ruu",
    "uu",
    "prabowo",
    "jokowi",
    "polri",
    "orba",
    "1998",
    "1965",
    "2024",
    "2025",
    "pppk",
    "pasal",
    "demokrasi",
    "oligarki",
    "pemerintahan",
    "pemerintah",
    "jabatan",
    "pejabat",
    "kekuasaan",
    "melakukan",
    "dilakukan",
    "pemimpin",
    "kepemimpinan",
    "masyarakat",
    "kebijakan",
    "keadilan",
    "perampasan",
    "korupsi",
    "koruptor",
    "mahasiswa",
    "reformasi",
    "undang",
} | SPECIAL_TOKENS | NEGATIONS | INTENSIFIERS | DOMAIN_TERMS | SENTIMENT_TERMS
PRESERVED_STOPWORDS = NEGATIONS | INTENSIFIERS | DOMAIN_TERMS | SENTIMENT_TERMS | {"mengapa", "sudah"}

SLANG_MAP = {
    "salahsatu": "salah satu",
    "adlah": "adalah",
    "dn": "dan",
    "spt": "seperti",
    "sebutkn": "sebutkan",
    "mndptkn": "mendapatkan",
    "bertanggungjawab": "bertanggung jawab",
    "yg": "yang",
    "ga": "tidak",
    "gak": "tidak",
    "nggak": "tidak",
    "ngga": "tidak",
    "ngak": "tidak",
    "gk": "tidak",
    "kagak": "tidak",
    "tdk": "tidak",
    "klo": "kalau",
    "kalo": "kalau",
    "gw": "saya",
    "gue": "saya",
    "gua": "saya",
    "ane": "saya",
    "lu": "kamu",
    "lo": "kamu",
    "loe": "kamu",
    "elu": "kamu",
    "dr": "dari",
    "dgn": "dengan",
    "dg": "dengan",
    "utk": "untuk",
    "krn": "karena",
    "karna": "karena",
    "jd": "jadi",
    "jdi": "jadi",
    "tp": "tapi",
    "trs": "terus",
    "trus": "terus",
    "bs": "bisa",
    "bgt": "banget",
    "bngt": "banget",
    "emg": "memang",
    "emang": "memang",
    "udah": "sudah",
    "sdh": "sudah",
    "blm": "belum",
    "blom": "belum",
    "org": "orang",
    "orng": "orang",
    "aja": "saja",
    "skrg": "sekarang",
    "skrng": "sekarang",
    "sampe": "sampai",
    "ampe": "sampai",
    "tau": "tahu",
    "gmn": "bagaimana",
    "gimana": "bagaimana",
    "knp": "kenapa",
    "kyk": "seperti",
    "kayak": "seperti",
    "kaya": "seperti",
    "cm": "hanya",
    "cuma": "hanya",
    "doang": "saja",
    "makin": "semakin",
    "pak": "bapak",
    "bkn": "bukan",
    "jgn": "jangan",
    "dah": "sudah",
    "udh": "sudah",
    "sm": "sama",
    "pake": "pakai",
    "pengen": "ingin",
    "makasih": "terima kasih",
    "makasi": "terima kasih",
    "thx": "terima kasih",
    "thanks": "terima kasih",
    "tq": "terima kasih",
    "ky": "seperti",
    "kek": "seperti",
    "ni": "ini",
    "nih": "ini",
    "sih": "",
    "dong": "",
    "min": "admin",
    "bro": "",
    "sis": "",
    "dpt": "dapat",
    "dapet": "dapat",
    "dlm": "dalam",
    "pd": "pada",
    "smua": "semua",
    "sbg": "sebagai",
    "hrs": "harus",
    "msih": "masih",
    "bbrp": "beberapa",
    "ttp": "tetap",
    "pinter": "pintar",
    "ngomong": "bicara",
    "ngurus": "urus",
    "ngatur": "atur",
    "ngotak": "otak",
    "ok": "baik",
    "okay": "baik",
    "fix": "pasti",
    "bener": "benar",
    "bnr": "benar",
    "beneran": "benar",
    "gini": "begini",
    "gitu": "begitu",
    "ginian": "begini",
    "begni": "begini",
    "bgini": "begini",
    "bgitu": "begitu",
    "nntn": "nonton",
    "nontonin": "nonton",
    "liat": "lihat",
    "liatin": "lihat",
    "ngeliat": "lihat",
    "bang": "",
    "bung": "",
    "mas": "",
    "mba": "",
    "mbak": "",
    "pakde": "",
    "aelah": "",
    "ajah": "saja",
    "ajh": "saja",
    "apaan": "apa",
    "bct": "bacot",
    "bacoott": "bacot",
    "bacoot": "bacot",
    "bacottt": "bacot",
    "dongo": "bodoh",
    "tololll": "tolol",
    "goblokk": "goblok",
    "bejat": "buruk",
    "ajur": "hancur",
    "ancur": "hancur",
    "bobrok": "buruk",
    "bobroknya": "buruk",
    "omon": "omong",
    "omon2": "omong omong",
    "koar2": "koar koar",
    "diem": "diam",
    "diem2": "diam diam",
    "dimanaa": "dimana",
    "konoha": "indonesia",
    "wakanda": "indonesia",
    "indo": "indonesia",
    "bowo": "prabowo",
    "dedy": "deddy",
    "deddy": "deddy",
    "dprri": "dpr ri",
    "uud": "uud",
    "uu": "uu",
    "ruu": "ruu",
    "se7": "setuju",
    "setujuu": "setuju",
    "gass": "gas",
    "gaskeun": "gas",
    "auto": "otomatis",
    "alhamdulillah": "syukur",
    "astaga": "kaget",
    "ngeri": "ngeri",
    "jg": "juga",
    "lg": "lagi",
    "tu": "itu",
    "mah": "",
    "bro": "",
    "bray": "",
    "min": "",
    "admin": "",
    "oke": "baik",
    "okey": "baik",
    "negri": "negeri",
    "rezim": "pemerintah",
    "milih": "pilih",
    "ngebahas": "bahas",
    "ngomongin": "bicara",
    "ngawur": "kacau",
    "dungu": "bodoh",
    "pinter": "pintar",
    "pintarrr": "pintar",
    "kadrun": "",
    "cebong": "",
    "buzzerp": "buzzer",
    "buzer": "buzzer",
    "buzzeer": "buzzer",
    "wakil2": "wakil wakil",
    "rakyat2": "rakyat rakyat",
    "s3": "",
    "s2": "",
    "s1": "",
    "p3k": "pppk",
    "g30s": "g30s",
    "g30spki": "g30s pki",
    "98": "1998",
    "98%": "98%",
    "kl": "kalau",
    "klw": "kalau",
    "klu": "kalau",
    "khl": "kalau",
    "lgi": "lagi",
    "msh": "masih",
    "dri": "dari",
    "sma": "sama",
    "ama": "sama",
    "tpi": "tapi",
    "gtu": "begitu",
    "mrk": "mereka",
    "byk": "banyak",
    "yng": "yang",
    "sya": "saya",
    "sy": "saya",
    "bpk": "bapak",
    "thn": "tahun",
    "th": "tahun",
    "ttg": "tentang",
    "bhw": "bahwa",
    "shg": "sehingga",
    "lbh": "lebih",
    "mlh": "malah",
    "ngk": "tidak",
    "gpp": "tidak apa apa",
    "gaada": "tidak ada",
    "bcra": "bicara",
    "tntg": "tentang",
    "ksjadian": "kejadian",
    "tragrdi": "tragedi",
    "bkal": "bakal",
    "dh": "sudah",
    "ja": "saja",
    "tr": "nanti",
    "fiks": "pasti",
    "rill": "real",
    "ril": "real",
    "ratt": "",
    "dor": "",
    "dora": "",
    "wd": "",
    "cuan": "",
    "president": "presiden",
    "country": "negara",
    "military": "militer",
    "civil": "sipil",
    "right": "hak",
    "rights": "hak",
    "human": "manusia",
    "story": "cerita",
    "soldier": "tentara",
    "war": "perang",
    "lord": "tuan",
    "greed": "rakus",
    "independent": "merdeka",
    "safe": "aman",
    "future": "masa depan",
    "health": "kesehatan",
    "safety": "keselamatan",
    "adlah": "adalah",
    "beraatu": "bersatu",
    "wrg": "warga",
    "kasib": "kasih",
    "ndji": "pandji",
    "salahsatu": "salah satu",
    "tentalatentalaan": "tentara tentaraan",
    "dodol": "bodoh",
    "lacur": "melacurkan",
    "mreka": "mereka",
    "cma": "cuma",
    "ktanya": "katanya",
    "kmren": "kemarin",
    "waklik": "wakil",
    "ndasmu": "kepalamu",
    "endasmuu": "kepalamu",
    "gitui": "begitu",
    "itui": "itu",
    "mikir": "berpikir",
    "ngerasain": "merasakan",
    "ngrasa": "merasa",
    "ngharap": "berharap",
    "ngharapin": "berharap",
    "brharap": "berharap",
    "brarti": "berarti",
    "smpe": "sampai",
    "smp": "sampai",
    "dinegara": "di negara",
    "dipemerintahan": "di pemerintahan",
    "disemua": "di semua",
    "diluar": "di luar",
    "diindonesia": "di indonesia",
    "dprri": "dpr ri",
    "bcra": "bicara",
    "tntg": "tentang",
    "brkuasa": "berkuasa",
    "duid": "duit",
    "duit": "uang",
    "jmlh": "jumlah",
    "krsi": "kursi",
    "dkit": "sedikit",
    "dikit": "sedikit",
    "cba": "coba",
    "knapa": "kenapa",
    "trnyata": "ternyata",
    "dsini": "di sini",
    "bgini": "begini",
    "bginilah": "beginilah",
    "bgaimana": "bagaimana",
    "klean": "kalian",
    "suangat": "sangat",
    "buanyak": "banyak",
    "bljr": "belajar",
    "dlu": "dulu",
    "bsa": "bisa",
    "psti": "pasti",
    "mabes": "markas besar",
    "drpd": "daripada",
    "magabut": "gabut",
    "rejeki": "rezeki",
    "milu": "pemilu",
    "family": "keluarga",
    "cause": "karena",
    "couse": "karena",
    "happen": "terjadi",
    "shit": "masalah",
    "wtf": "",
    "ii": "",
    "yaa": "",
    "yah": "",
    "ad": "ada",
    "aj": "saja",
    "ny": "nya",
    "bnyk": "banyak",
    "mnding": "mending",
    "lbih": "lebih",
    "mpai": "sampai",
    "smpai": "sampai",
    "taun": "tahun",
    "olh": "oleh",
    "blh": "boleh",
    "cheos": "kekacauan",
    "ngg": "tidak",
    "nga": "tidak",
    "egk": "tidak",
    "gx": "tidak",
    "kgk": "tidak",
    "bkin": "bikin",
    "nnti": "nanti",
    "ntr": "nanti",
    "lgnsung": "langsung",
    "poko": "pokok",
    "pokonya": "pokoknya",
    "jngan": "jangan",
    "janngann": "jangan",
    "ngomomg": "bicara",
    "ngurusin": "urus",
    "ngeliat": "lihat",
    "nyiptain": "menciptakan",
    "dipikirnya": "dipikir",
    "dipikirannya": "pikirannya",
    "seandai": "seandainya",
    "selere": "selera",
    "dikomporin": "diprovokasi",
    "dimanfaatin": "dimanfaatkan",
    "dibodohin": "dibodohi",
    "ngebayangin": "membayangkan",
    "nabrak": "melanggar",
    "mengusulkn": "mengusulkan",
    "pertanya": "pertanyaan",
    "anya": "",
    "ngibul": "bohong",
    "guntung": "untung",
    "prajuri": "prajurit",
    "nganggur": "menganggur",
    "instutusi": "institusi",
    "nambahin": "menambah",
    "ngelindungin": "melindungi",
    "ngilangin": "menghilangkan",
    "nutup": "menutup",
    "becus": "mampu",
    "dinas2": "dinas dinas",
    "rtkades": "rt kades",
    "pejababat": "pejabat",
    "dimksd": "dimaksud",
    "dibwh": "di bawah",
    "sbab": "sebab",
    "menbunuh": "membunuh",
    "mengalamix": "mengalami",
    "hny": "hanya",
    "cmn": "cuma",
    "doank": "saja",
    "kpn": "kapan",
    "mjd": "menjadi",
    "tdi": "tadi",
    "hrg": "harga",
    "byr": "bayar",
    "scr": "secara",
    "mcm": "macam",
    "sll": "selalu",
    "msk": "masuk",
    "sebutkn": "sebutkan",
    "bgi": "bagi",
    "bginilah": "beginilah",
    "bgaimana": "bagaimana",
    "bng": "bang",
    "bgs": "bagus",
    "bru": "baru",
    "buanyak": "banyak",
    "suangat": "sangat",
    "drpd": "daripada",
    "psti": "pasti",
    "bsa": "bisa",
    "mabes": "markas besar",
    "magabut": "gabut",
    "rejeki": "rezeki",
    "milu": "pemilu",
    "brkuasa": "berkuasa",
    "duid": "uang",
    "brharap": "berharap",
    "ngharapin": "berharap",
    "ngrasa": "merasa",
    "ngrasain": "merasakan",
    "brarti": "berarti",
    "smpe": "sampai",
    "smp": "sampai",
    "kmren": "kemarin",
    "kmrn": "kemarin",
    "ktanya": "katanya",
    "gitui": "begitu",
    "itui": "itu",
    "ndasmu": "kepalamu",
    "endasmuu": "kepalamu",
    "waklik": "wakil",
    "cma": "cuma",
    "mreka": "mereka",
    "beraatu": "bersatu",
    "wrg": "warga",
    "kasib": "kasih",
    "ndji": "pandji",
    "tentalatentalaan": "tentara tentaraan",
    "lacur": "melacurkan",
    "dodol": "bodoh",
    "adl": "adalah",
    "akn": "akan",
    "blg": "bilang",
    "bnr": "benar",
    "brp": "berapa",
    "bwh": "bawah",
    "dng": "dengan",
    "dngan": "dengan",
    "dngn": "dengan",
    "dmn": "dimana",
    "gwe": "saya",
    "jls": "jelas",
    "jng": "jangan",
    "jngn": "jangan",
    "jmn": "jaman",
    "jamanborde": "jaman orde",
    "kga": "tidak",
    "koq": "mengapa",
    "kya": "seperti",
    "lha": "lah",
    "mmg": "memang",
    "mmng": "memang",
    "mngkin": "mungkin",
    "mgkin": "mungkin",
    "mgkn": "mungkin",
    "pke": "pakai",
    "pny": "punya",
    "sdg": "sedang",
    "sdr": "saudara",
    "sgt": "sangat",
    "sja": "saja",
    "skg": "sekarang",
    "skr": "sekarang",
    "spy": "supaya",
    "tks": "terima kasih",
    "drpada": "daripada",
    "krna": "karena",
    "tsb": "tersebut",
    "fakum": "vakum",
    "nggakii": "tidak",
    "nggakiii": "tidak",
    "naekoba": "naik coba",
    "mjatuhkan": "menjatuhkan",
    "mresahkan": "meresahkan",
    "sbb": "sebab",
    "pdhl": "padahal",
    "lgsg": "langsung",
    "blm": "belum",
    "sblm": "sebelum",
    "stlh": "setelah",
    "sdh": "sudah",
    "tdk": "tidak",
    "krn": "karena",
    "utk": "untuk",
    "dlm": "dalam",
    "jg": "juga",
    "pd": "pada",
    "jd": "jadi",
    "org": "orang",
    "rkyt": "rakyat",
    "qt": "kita",
    "pgn": "ingin",
    "kwn": "kawan",
    "tgl": "tinggal",
    "sngt": "sangat",
    "msyrkt": "masyarakat",
    "dzolim": "zalim",
    "dzalim": "zalim",
    "zolim": "zalim",
    "zholim": "zalim",
    "unt": "untuk",
    "wkt": "waktu",
    "gni": "begini",
    "gituu": "begitu",
    "bg": "bang",
    "kpd": "kepada",
    "thd": "terhadap",
    "thdp": "terhadap",
    "km": "kamu",
    "mo": "mau",
    "mw": "mau",
    "ato": "atau",
    "atw": "atau",
    "kok": "mengapa",
    "ud": "sudah",
    "ndak": "tidak",
    "ngerti": "mengerti",
    "ngapa": "apa",
    "ngapain": "sedang apa",
    "plong": "kosong",
    "plongo": "kosong",
    "planga": "yang",
    "ndeso": "kampung",
    "orba": "orde baru",
    "bqn": "bikin",
    "brita": "berita",
    "jlz": "jelas",
    "trz": "terus",
    "hrz": "harus",
    "zmn": "zaman",
    "bqnyak": "banyak",
    "terjadii": "terjadi",
    "beliaui": "beliau",
    "mnduduki": "menduduki",
    "walapaun": "walaupun",
    "walaupaun": "walaupun",
    "inndonesia": "indonesia",
    "inonesia": "indonesia",
    "froxy": "proxy",
    "siplongo": "si kosong",
    "plongonya": "kosongnya",
    "repormasi": "reformasi",
    "hawatir": "khawatir",
    "wrga": "warga",
    "terkorup": "terkorupsi",
    "komunisi": "komunis",
    "kmrin": "kemarin",
    "dmna": "dimana",
    "krismon": "krisis moneter",
    "gilaa": "gila",
    "hadeehh": "hadeh",
    "gaes": "teman",
    "tengok": "lihat",
    "nggakkk": "tidak",
    "nggakk": "tidak",
    "nggap": "tidak apa",
    "kalu": "kalau",
    "propensi": "provinsi",
    "maerdeka": "merdeka",
    "sediri": "sendiri",
    "adlh": "adalah",
    "bhkn": "bahkan",
    "yqng": "yang",
    "ngabyak": "banyak",
    "mkn": "makan",
    "bgmn": "bagaimana",
    "cpt": "cepat",
    "ngmg": "bicara",
    "nnt": "nanti",
    "bgd": "banget",
    "bhn": "bahan",
    "bhs": "bahas",
    "blkg": "belakang",
    "blng": "bilang",
    "bwt": "buat",
    "dtg": "datang",
    "dtng": "datang",
    "ggr": "gara gara",
    "hdp": "hidup",
    "jwb": "jawab",
    "jwbn": "jawaban",
    "kdg": "kadang",
    "kmn": "kemana",
    "mdh": "mudah",
    "mntp": "mantap",
    "mslh": "masalah",
    "ndk": "tidak",
    "prnh": "pernah",
    "slh": "salah",
    "slm": "salam",
    "tbtb": "tiba tiba",
    "tjd": "terjadi",
    "tlg": "tolong",
    "trsrh": "terserah",
    "pntg": "penting",
    "sdikit": "sedikit",
    "pekerjan": "pekerjaan",
    "dengn": "dengan",
    "hbis": "habis",
    "pulng": "pulang",
    "komntr": "komentar",
    "seblum": "sebelum",
    "istirht": "istirahat",
    "pnegalmn": "pengalaman",
    "oembahasan": "pembahasan",
    "shrusx": "seharusnya",
    "secepatx": "secepatnya",
    "mlah": "malah",
    "knpa": "kenapa",
    "klau": "kalau",
    "jga": "juga",
    "bner": "benar",
    "emng": "memang",
    "donk": "dong",
    "ntn": "nonton",
    "abis": "habis",
    "orng": "orang",
    "gmna": "bagaimana",
    "bnyak": "banyak",
    "skli": "sekali",
    "jln": "jalan",
    "klk": "kalau",
    "sjk": "sejak",
    "sjt": "juta",
    "drp": "dpr",
    "sth": "tahun",
    "tkt": "takut",
    "trjd": "terjadi",
    "ttng": "tentang",
    "lngsg": "langsung",
    "donx": "dong",
    "kn": "kan",
    "bisaa": "bisa",
    "pdahal": "padahal",
    "prmerintah": "pemerintah",
    "dmana": "dimana",
    "sluruh": "seluruh",
    "zdolim": "zalim",
    "tnii": "tni",
    "prabowoi": "prabowo",
    "pemerintahi": "pemerintah",
    "terwakilii": "terwakili",
    "isinyaii": "isinya",
    "kesitui": "kesitu",
    "samaii": "sama",
    "kemarinii": "kemarin",
    "suriahi": "suriah",
    "rakyati": "rakyat",
    "inii": "ini",
    "bangi": "bang",
    "sihi": "sih",
    "apai": "apa",
    "apaii": "apa",
    "kahi": "kah",
    "manai": "mana",
    "sipili": "sipil",
    "militeri": "militer",
    "dpri": "dpr",
    "disahkani": "disahkan",
    "negarai": "negara",
    "koruptori": "koruptor",
    "korupsii": "korupsi",
    "polrii": "polri",
    "indonesiai": "indonesia",
    "reformasii": "reformasi",
    "demokrasii": "demokrasi",
    "otaki": "otak",
    "politiki": "politik",
    "bisai": "bisa",
    "yai": "ya",
    "tohi": "toh",
    "toi": "toh",
    "koruptot": "koruptor",
    "bernyas": "berantas",
    "keseringen": "keseringan",
    "dipake": "dipakai",
    "dengernya": "mendengarnya",
    "menlanjutkan": "melanjutkan",
    "iggbii": "1998",
    "terimakasih": "terima kasih",
    "masarakat": "masyarakat",
    "sprti": "seperti",
    "sperti": "seperti",
    "maen": "main",
    "koropsi": "korupsi",
    "koroptor": "koruptor",
    "ketauan": "ketahuan",
    "smoga": "semoga",
    "prampasan": "perampasan",
    "dengerin": "mendengarkan",
    "skarang": "sekarang",
    "brani": "berani",
    "sistim": "sistem",
    "slalu": "selalu",
    "sndiri": "sendiri",
    "perduli": "peduli",
    "bangett": "banget",
    "mensahkan": "mengesahkan",
    "disyahkan": "disahkan",
    "alusista": "alutsista",
    "kawatir": "khawatir",
    "munkin": "mungkin",
    "banhsat": "bangsat",
    "t3mb4k": "tembak",
    "m4ti": "mati",
    "fuvk": "profanity_en",
    "jakpot": "jackpot",
    "lapanganpekerjaan": "lapangan pekerjaan",
    "berhentibayarpajak": "berhenti bayar pajak",
    "merdekaindonesiaantikorupsi": "merdeka indonesia anti korupsi",
    "pertanyaannyaiapakah": "pertanyaannya apakah",
    "dipertanyakanibagi": "dipertanyakan bagi",
    "haruusnyauutniituutidaperluu": "harusnya uu_tni itu tidak perlu",
    "yangpalingperluusijokowidan": "yang paling perlu jokowi dan",
    "agustotomenawarkan": "",
    "aerbbmenawarkan": "",
    "hiduprakyatyangmelawan": "hidup rakyat yang melawan",
    "hukummatikoruptor": "hukum mati koruptor",
    "hukummatiparakoruptor": "hukum mati para koruptor",
    "indonesiahancur": "indonesia hancur",
    "indonesiamakingelap": "indonesia makin gelap",
    "hidupkanrevolusi": "hidupkan revolusi",
    "birokrasipemerintah": "birokrasi pemerintah",
    "amnahrakyatyacamkan": "amanah rakyat camkan",
    "apalagimahasiswanya": "apalagi mahasiswanya",
    "harusbergerakturuntangan": "harus bergerak turun tangan",
    "dideklarasikani": "dideklarasikan",
    "dipersenjatakani": "dipersenjatakan",
    "hayamenjalankan": "hanya menjalankan",
    "hinggaakademidu": "hingga akademi",
    "iniseutuhyamilik": "ini seutuhnya milik",
    "itubselalubwaspada": "itu selalu waspada",
    "kekuatannpolitik": "kekuatan politik",
    "keluarganyadipindahkandari": "keluarganya dipindahkan dari",
    "asuusasuukwwkwk": "asu",
    "kewarganegaraani": "kewarganegaraan",
    "korupsidilindungi": "korupsi dilindungi",
    "masyarakatbiasa": "masyarakat biasa",
    "lakuiniikorupsi": "pelaku ini korupsi",
    "pls": "tolong",
    "plzz": "tolong",
    "pda": "pada",
    "bkl": "bakal",
    "blk": "balik",
    "blkng": "belakang",
    "bnyj": "banyak",
    "kndsi": "kondisi",
    "krpsi": "korupsi",
    "shrsny": "seharusnya",
    "prmpsn": "perampasan",
    "bknnya": "bukannya",
    "pengen": "ingin",
    "pngen": "ingin",
    "pnjr": "penjara",
    "jbtn": "jabatan",
    "senjsta": "senjata",
    "negaralain": "negara lain",
    "lsg": "langsung",
    "lsng": "langsung",
    "lmyn": "lumayan",
    "mksd": "maksud",
    "mksdny": "maksudnya",
    "mls": "malas",
    "mndng": "mending",
    "ngmng": "bicara",
    "nth": "entah",
    "ktny": "katanya",
    "kykny": "kayaknya",
    "lgy": "lagi",
    "hbs": "habis",
    "hdr": "hadir",
    "dkt": "dekat",
    "dmk": "demikian",
    "flm": "film",
    "gbs": "tidak bisa",
    "gkpp": "tidak apa apa",
    "jgnkn": "jangankan",
    "ksh": "kasih",
    "ktm": "ketemu",
    "lht": "lihat",
    "mdhn": "mudah mudahan",
    "mnrt": "menurut",
    "mnt": "minta",
    "rkyat": "rakyat",
    "sbnr": "sebenarnya",
    "sblumy": "sebelumnya",
    "trttup": "tertutup",
    "ahiry": "akhirnya",
    "llos": "lolos",
}

MIXED_ALNUM_MAP = {
    "2x": "",
    "v2": "",
    "g2": "",
    "2th": "tahun",
    "2thn": "tahun",
    "2tthn": "tahun",
    "2oth": "tahun",
    "2othn": "tahun",
    "2stahun": "tahun",
    "2etahun": "tahun",
    "e2tahun": "tahun",
    "2o29golputserentak": "golput serentak",
    "2029golputserentak": "golput serentak",
    "g30spki": "g30s pki",
    "p3k": "pppk",
}

REDUPLICATION_MAP = {
    "orang2": "orang orang",
    "org2": "orang orang",
    "bener2": "benar benar",
    "benar2": "benar benar",
    "undang2": "undang undang",
    "anak2": "anak anak",
    "pejabat2": "pejabat pejabat",
    "apa2": "apa apa",
    "siap2": "siap siap",
    "hati2": "hati hati",
    "tiba2": "tiba tiba",
    "diam2": "diam diam",
    "diem2": "diam diam",
    "lama2": "lama lama",
    "baik2": "baik baik",
    "gara2": "gara gara",
    "kira2": "kira kira",
    "sama2": "sama sama",
    "sehat2": "sehat sehat",
    "masing2": "masing masing",
    "bisa2": "bisa bisa",
    "pura2": "pura pura",
    "dikit2": "sedikit sedikit",
    "semena2": "semena mena",
    "antek2": "antek antek",
    "bagi2": "bagi bagi",
    "rata2": "rata rata",
    "buru2": "buru buru",
    "jelas2": "jelas jelas",
    "akhir2": "akhir akhir",
    "dimana2": "dimana mana",
    "jangan2": "jangan jangan",
    "sembunyi2": "sembunyi sembunyi",
    "cepet2": "cepat cepat",
    "jauh2": "jauh jauh",
    "kasus2": "kasus kasus",
    "teriak2": "teriak teriak",
    "aneh2": "aneh aneh",
    "demo2": "demo demo",
    "kata2": "kata kata",
    "oknum2": "oknum oknum",
    "seolah2": "seolah olah",
    "tikus2": "tikus tikus",
    "gila2an": "gila gila",
    "terang2an": "terang terang",
    "besar2an": "besar besar",
    "mati2an": "mati mati",
    "satu2nya": "satu satu",
    "ujung2nya": "ujung ujung",
    "besok2": "besok besok",
    "teman2": "teman teman",
    "sipil2": "sipil sipil",
    "isu2": "isu isu",
    "aman2": "aman aman",
    "mana2": "mana mana",
    "kebijakan2": "kebijakan kebijakan",
    "main2": "main main",
    "lembaga2": "lembaga lembaga",
    "jgn2": "jangan jangan",
    "maling2": "maling maling",
    "lain2": "lain lain",
    "ikut2an": "ikut ikut",
    "jendral2": "jendral jendral",
    "lagi2": "lagi lagi",
    "sangat2": "sangat sangat",
    "abal2": "abal abal",
    "bertahun2": "bertahun tahun",
    "cita2": "cita cita",
    "boro2": "boro boro",
    "mafia2": "mafia mafia",
    "mentah2": "mentah mentah",
    "video2": "video video",
    "kemana2": "kemana mana",
    "sia2": "sia sia",
    "bagus2": "bagus bagus",
    "bersih2": "bersih bersih",
    "satu2": "satu satu",
    "korupsi2": "korupsi korupsi",
    "konten2": "konten konten",
    "bangga2in": "bangga bangga",
    "bangga2kan": "bangga bangga",
    "besar2kan": "besar besar",
    "disah2kan": "sahkan sahkan",
    "sah2kan": "sahkan sahkan",
    "nakut2in": "takut takut",
    "dibodoh2i": "bodoh bodoh",
    "dibodoh2in": "bodoh bodoh",
    "ngebodoh2i": "bodoh bodoh",
    "mengagung2kan": "agung agung",
    "agung2kan": "agung agung",
    "pecah2in": "pecah pecah",
    "dipanas2in": "panas panas",
    "bodoh2in": "bodoh bodoh",
    "hambur2in": "hambur hambur",
    "iya2in": "iya iya",
    "malu2in": "malu malu",
    "tangkep2in": "tangkap tangkap",
    "jelek2in": "jelek jelek",
    "musuh2x": "musuh musuh",
    "anak2x": "anak anak",
    "dosa2x": "dosa dosa",
    "dibunuh2in": "bunuh bunuh",
    "dipecat2in": "pecat pecat",
    "dimana2x": "dimana mana",
    "dimain2nin": "main main",
    "ujung2mya": "ujung ujung",
    "hancur2lah": "hancur hancur",
    "aneh2aja": "aneh aneh",
    "suka2mu": "suka suka",
    "siap2lah": "siap siap",
    "bela2in": "bela bela",
    "ogah2n": "ogah ogah",
    "blak2kan": "blak blak",
    "abis2in": "habis habis",
    "disama2in": "sama sama",
    "tutup2i": "tutup tutup",
    "bohong2in": "bohong bohong",
    "ditutup2i": "tutup tutup",
    "goblok2in": "goblok goblok",
    "main2dengan": "main main",
    "baik2saja": "baik baik",
    "bapak2tni": "bapak bapak tni",
    "gembor2in": "gembor gembor",
    "antek2nyaa": "antek antek",
    "ugal2lan": "ugal ugal",
    "teman2nga": "teman teman",
    "kroni2x": "kroni kroni",
    "sebaik2x": "baik baik",
}

ENGLISH_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "but", "by", "for",
    "from", "got", "had", "has", "have", "he", "her", "his", "i", "if", "in",
    "is", "it", "me", "my", "not", "of", "on", "or", "our", "she", "so",
    "that", "the", "their", "them", "they", "this", "to", "was", "we", "were",
    "will", "with", "you", "your", "gonna", "wanna", "take", "listen", "longer",
    "already", "doomed", "account", "second", "big", "dislike", "any", "income",
    "spin", "login", "child", "its", "just", "like", "coupe", "etat",
    "man", "usa", "standup", "artist", "hero", "speak", "these", "crying",
    "amazed", "how", "opinion", "want", "let", "know", "release", "public",
    "please", "always", "pray", "assure", "dont", "care", "about", "guy",
    "feeling", "close", "door", "stupidity", "fuck", "nada", "nothing", "none",
    "thats", "why", "give", "flying", "what", "make", "think", "time", "process",
    "march", "term", "multiple", "riots", "currently", "revising", "constitution",
    "personnel", "intervene", "matters", "screams", "red", "flags", "everywhere",
    "first", "comment", "being", "destroyed", "more", "due", "sumatra", "java",
    "smart", "people", "all",
}

CUSTOM_STOPWORDS = {
    "nya", "kan", "lah", "sih", "dong", "nih", "ya", "si", "tuh", "mah",
    "bang", "bung", "mas", "mbak", "mba", "pak", "bapak", "bro", "sis", "min",
    "admin", "deh", "dong", "pun", "bin", "aja", "saja", "url_token",
    "user_mention", "to", "not", "bullshit", "child", "its", "just", "like",
    "coupe", "etat", "hmadt", "toto", "jep", "jepe", "ii", "yaa", "yah",
    "btw", "noh", "lho", "loh", "eh", "ehh", "ah", "hmm", "hehe", "wkwk",
    "wkwkw", "wkwkwk", "wkwkwkw", "wkwkwkwk", "wkwkk", "wkwkkw", "wkwkkwkw",
    "wkwkwwk", "wkk", "wkkw", "haha", "hahaha",
    "jadi", "kalau", "karena", "dari", "sama", "apa", "mau", "semua", "orang",
    "baru", "banyak", "sekarang", "buat", "kamu", "biar", "supaya", "serta",
    "via", "begitu", "begini", "sana", "sini", "situ", "lalu", "kemudian",
    "selanjutnya", "namun", "tetapi", "tapi", "sedangkan", "padahal",
    "sementara", "sebenarnya", "seharusnya", "bisa", "dapat", "mampu", "boleh",
    "harus", "pasti", "tentu", "mungkin", "atau", "maupun", "apakah", "kapan",
    "dimana", "kemana", "bagaimana", "kenapa", "siapa", "para", "sang", "per",
    "setiap", "masing", "sendiri", "antara", "yakni",
    "sebab", "sehingga", "hanya", "hampir", "agak", "cukup", "semoga",
    "kiranya", "alangkah", "sungguh", "kelak", "nanti", "dulu", "terus",
}
LEETSPEAK_TABLE = str.maketrans({"0": "o", "1": "i", "2": "z", "3": "e", "4": "a", "5": "s", "7": "t", "8": "b", "9": "g"})
CONFUSABLE_TABLE = str.maketrans({
    "?": "A", "?": "a", "?": "B", "?": "E", "?": "e", "?": "K", "?": "M",
    "?": "H", "?": "O", "?": "o", "?": "P", "?": "p", "?": "C", "?": "c",
    "?": "T", "?": "X", "?": "x", "?": "I", "?": "i", "?": "J", "?": "j",
    "?": "Y", "?": "O", "?": "o", "?": "A", "?": "a", "?": "A", "?": "I",
})
NUMERIC_KEEP_TOKENS = {"1945", "1965", "1998", "2024", "2025", "2029", "2030", "02"}

EMOJI_GROUPS = {
    "emo_laugh": {"😂", "🤣", "😆", "😹"},
    "emo_sad": {"😢", "😭", "🥲", "😞", "😔"},
    "emo_angry": {"😡", "🤬", "😠"},
    "emo_pos": {"👍", "✅", "❤️", "❤", "🔥", "💪", "🙏"},
    "emo_neg": {"👎", "❌", "💩"},
    "emo_think": {"🤔", "🧐"},
}

URL_RE = re.compile(r"(?i)\b(?:https?://|www\.)\S+")
HTML_TAG_RE = re.compile(r"<[^>]+>")
TIMESTAMP_RE = re.compile(r"(?<!\d)\d{1,2}:\d{2}(?::\d{2})?(?!\d)")
MENTION_RE = re.compile(r"(?<!\w)@[\w.-]+")
HASHTAG_RE = re.compile(r"#([\w]+)")
ZERO_WIDTH_RE = re.compile(r"[\u200b-\u200f\u2060\ufeff]")
WHITESPACE_RE = re.compile(r"\s+")
ELONGATED_RE = re.compile(r"(?i)([a-z])\1{2,}")
CAMEL_BOUNDARY_RE = re.compile(r"(?<=[a-z])(?=[A-Z])")
RAW_TOKEN_RE = regex.compile(r"[\p{L}\p{N}_]+")
ALL_CAPS_TOKEN_RE = regex.compile(r"\b\p{Lu}{2,}\b")
SPACED_LETTER_RE = regex.compile(r"(?<![\p{L}\p{N}])(?:[\p{L}\p{N}]\s+){2,}[\p{L}\p{N}](?![\p{L}\p{N}])")
REPEATED_SHORT_TOKEN_RE = re.compile(r"(?i)^([a-z]{2,})\d{1,2}$")
REDUPLICATION_SUFFIX_RE = re.compile(r"(?i)^([a-z]{3,})2(?:nya|an|in|kan|lah|x|mya|mu|ku)?$")
SPAM_TOKEN_RE = re.compile(r"(?i)(togel|slot|gacor|gacir|gachor|gchor|gachr|casino|bet|judi|judol|cuan|koislot|agustoto|agustto|agustot|ahmadtoto|ahmadt|hmadtoto|hmadt|doratoto|dewadora|dwadra|wadora|dorat|dora\d*|aerbb|aero\d*|thor\d*|thoreii|maxwin|mekwin|ro88|o88|ambil4d|mandalika\d*|jptogel\d*|miya\d+|ratt|jp|wd|withdraw|depo|deposit|bonus|jackpot|jakpot|jepe|jep)")
LAUGH_TOKEN_RE = re.compile(r"(?i)([wk]{4,}|[ha]{6,}|(?:awo?k){2,}.*|(wk)+w*|(kw)+k*w*|w+k+w*k+|(ah)+a*|ha(ha)+|he(he)+)")
SPAM_COMMENT_RE = re.compile(r"(?i)(togel|slot|gacor|gacir|gachor|gchor|casino|judi|judol|koislot|agustoto|agustto|ahmadtoto|hmadtoto|doratoto|dewadora|dwadra|wadora|mandalika|jptogel|ambil4d|aerbb|aero\d*|thor\d*|thoreii|maxwin|mekwin|jackpot|jakpot|deposit|depo|withdraw|wd|jp|jepe|jep|menang tiap hari|bonus)")

STOPWORDS = (set(StopWordRemoverFactory().get_stop_words()) | ENGLISH_STOPWORDS | CUSTOM_STOPWORDS) - PRESERVED_STOPWORDS
_STEMMER = None
_STEM_CACHE: dict[str, str] = {}

PREPROCESS_SCHEMA = StructType(
    [
        StructField(FINAL_TEXT_COLUMN, StringType(), False),
        StructField("tokens", ArrayType(StringType(), False), False),
        StructField("normalized_token_count", IntegerType(), False),
        StructField("raw_char_count", IntegerType(), False),
        StructField("raw_word_count", IntegerType(), False),
        StructField("emoji_count", IntegerType(), False),
        StructField("url_count", IntegerType(), False),
        StructField("mention_count", IntegerType(), False),
        StructField("hashtag_count", IntegerType(), False),
        StructField("exclamation_count", IntegerType(), False),
        StructField("question_count", IntegerType(), False),
        StructField("uppercase_ratio", DoubleType(), False),
        StructField("is_all_caps", BooleanType(), False),
        StructField("contains_negation", BooleanType(), False),
        StructField("contains_domain_terms", BooleanType(), False),
        StructField("contains_profanity", BooleanType(), False),
        StructField("domain_term_count", IntegerType(), False),
        StructField("profanity_count", IntegerType(), False),
    ]
)
REPORT_SCHEMA = StructType(
    [
        StructField("run_id", StringType(), False),
        StructField("processed_at", StringType(), False),
        StructField("input_collection", StringType(), False),
        StructField("output_collection", StringType(), False),
        StructField("total_rows_input", LongType(), False),
        StructField("total_rows_output", LongType(), False),
        StructField("total_unique_videos", LongType(), False),
        StructField("total_unique_authors", LongType(), False),
        StructField("missing_text_count", LongType(), False),
        StructField("duplicate_comment_id_count", LongType(), False),
        StructField("duplicate_text_original_count", LongType(), False),
        StructField("duplicate_text_count", LongType(), False),
        StructField("short_comments_count", LongType(), False),
        StructField("long_comments_count", LongType(), False),
        StructField("comments_with_emoji_count", LongType(), False),
        StructField("comments_with_url_count", LongType(), False),
        StructField("comments_with_mention_count", LongType(), False),
        StructField("comments_with_hashtag_count", LongType(), False),
        StructField("label_distribution_json", StringType(), False),
        StructField("validation_metrics_json", StringType(), False),
        StructField("top_tokens_before_json", StringType(), False),
        StructField("top_tokens_after_json", StringType(), False),
        StructField("total_profanity", LongType(), False),
        StructField("total_domain_terms", LongType(), False),
        StructField("empty_text_final_count", LongType(), False),
        StructField("warnings", ArrayType(StringType(), False), False),
    ]
)


def required_env(name: str) -> str:
    value = os.getenv(name, "").strip()
    if not value:
        raise ValueError(f"Konfigurasi wajib belum diisi: {name}")
    return value


def env_bool(name: str, default: bool = False) -> bool:
    return os.getenv(name, str(default)).strip().lower() in {"1", "true", "yes", "y"}


def env_int(name: str, default: int) -> int:
    value = os.getenv(name, "").strip()
    if not value:
        return default
    try:
        parsed = int(value)
    except ValueError as exc:
        raise ValueError(f"Konfigurasi {name} harus berupa integer.") from exc
    return parsed if parsed > 0 else default


def collapse_spaced_letters(match: regex.Match) -> str:
    groups = re.split(r"\s{2,}", match.group(0).strip())
    return " ".join("".join(group.split()) for group in groups if group.strip())


def basic_clean_text(text: Any) -> str:
    cleaned = ftfy.fix_text(html.unescape(str(text or "")))
    cleaned = HTML_TAG_RE.sub(" ", cleaned)
    cleaned = unicodedata.normalize("NFKC", cleaned).translate(CONFUSABLE_TABLE)
    cleaned = cleaned.replace("\\n", " ").replace("\\r", " ").replace("\\t", " ")
    cleaned = ZERO_WIDTH_RE.sub("", cleaned)
    cleaned = TIMESTAMP_RE.sub(" ", cleaned)
    cleaned = SPACED_LETTER_RE.sub(collapse_spaced_letters, cleaned)
    cleaned = cleaned.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    return WHITESPACE_RE.sub(" ", cleaned).strip()


LEGAL_PATTERNS = [
    (re.compile(r"(?i)\buud\s*nri\s*1945\b"), " uud_nri_1945 "),
    (re.compile(r"(?i)\buud\s*1945\b"), " uud_1945 "),
    (re.compile(r"(?i)\bruu\s+tni\b"), " ruu_tni "),
    (re.compile(r"(?i)\buu\s+tni\b"), " uu_tni "),
    (re.compile(r"(?i)\buud\s+tni\b"), " uu_tni "),
    (re.compile(r"(?i)\bruu\s+polri\b"), " ruu_polri "),
    (re.compile(r"(?i)\buu\s+polri\b"), " uu_polri "),
    (re.compile(r"(?i)\buud\s+polri\b"), " uu_polri "),
    (re.compile(r"(?i)\bruu\s+perampasan\s+aset\b"), " ruu_perampasan_aset "),
    (re.compile(r"(?i)\buu\s+perampasan\s+aset\b"), " uu_perampasan_aset "),
    (re.compile(r"(?i)\buud\s+perampasan\s+aset\b"), " uu_perampasan_aset "),
]
POLITICAL_02_CONTEXT_RE = re.compile(r"(?i)\b(paslon|pilih|pemilih|menang|kalah|prabowo|gibran|jokowi|pilpres)\b")


def normalize_legal_terms(text: str) -> str:
    normalized = text
    normalized = re.sub(r"(?i)\b58\s*(?:%|persen)\b", " angka_58_persen ", normalized)
    normalized = re.sub(r"(?i)\b98\b", " 1998 ", normalized)
    if POLITICAL_02_CONTEXT_RE.search(normalized):
        normalized = re.sub(r"(?<!\d)02(?!\d)", " paslon_02 ", normalized)
    for pattern, replacement in LEGAL_PATTERNS:
        normalized = pattern.sub(replacement, normalized)
    return WHITESPACE_RE.sub(" ", normalized).strip()


def ascii_fold(value: str) -> str:
    normalized = unicodedata.normalize("NFKD", value.translate(CONFUSABLE_TABLE))
    return normalized.encode("ascii", "ignore").decode("ascii")


def compact_for_spam(value: str) -> str:
    folded = ascii_fold(value).casefold().translate(LEETSPEAK_TABLE)
    return re.sub(r"[^a-z0-9]+", "", folded)


def is_spam_comment(value: str) -> bool:
    folded = ascii_fold(value).casefold()
    compact = compact_for_spam(value)
    return bool(SPAM_COMMENT_RE.search(folded) or SPAM_COMMENT_RE.search(compact))


def count_terms(tokens: list[str], terms: set[str]) -> int:
    return sum(token in terms for token in tokens)


def emoji_token(character: str) -> str:
    for token, characters in EMOJI_GROUPS.items():
        if character in characters:
            return token
    return "emo_other"


def replace_emojis(text: str) -> str:
    for item in emoji.emoji_list(text):
        text = text.replace(item["emoji"], f" {emoji_token(item['emoji'])} ")
    return text


def expand_hashtag(value: str) -> str:
    separated = CAMEL_BOUNDARY_RE.sub(" ", value).replace("_", " ").casefold()
    for combined, expanded in (
        ("2029golputserentak", "golput serentak"),
        ("2o29golputserentak", "golput serentak"),
        ("tolakruutni", "tolak ruu tni"),
        ("tolakrevisiuutni", "tolak revisi uu tni"),
        ("revisiuutni", "revisi uu tni"),
        ("ruutni", "ruu tni"),
        ("uutni", "uu tni"),
    ):
        separated = separated.replace(combined, expanded)
    return separated


def normalize_token(token: str, slang: dict[str, str]) -> list[str]:
    token = ascii_fold(token).casefold().strip("_")
    if not token:
        return []
    if SPAM_TOKEN_RE.search(token) or LAUGH_TOKEN_RE.fullmatch(token):
        return []
    if token in SPECIAL_TOKENS or token in NUMERIC_KEEP_TOKENS:
        return [token]
    if token in MIXED_ALNUM_MAP:
        return RAW_TOKEN_RE.findall(MIXED_ALNUM_MAP[token])
    if token in slang:
        return RAW_TOKEN_RE.findall(slang[token])
    if token in REDUPLICATION_MAP:
        return RAW_TOKEN_RE.findall(REDUPLICATION_MAP[token])
    if any(character.isalpha() for character in token) and any(character.isdigit() for character in token):
        if re.fullmatch(r"\d{1,2}thn?", token):
            return ["tahun"]
        if re.fullmatch(r"\d+[a-z]+", token):
            return []
        repeated_match = REPEATED_SHORT_TOKEN_RE.match(token)
        redup_match = REDUPLICATION_SUFFIX_RE.match(token)
        if redup_match:
            base = redup_match.group(1)
            normalized_base = slang.get(base, base)
            token = f"{normalized_base} {normalized_base}"
        elif repeated_match:
            token = repeated_match.group(1)
        else:
            translated = token.translate(LEETSPEAK_TABLE)
            if translated in MIXED_ALNUM_MAP:
                return RAW_TOKEN_RE.findall(MIXED_ALNUM_MAP[translated])
            if SPAM_TOKEN_RE.search(translated) or LAUGH_TOKEN_RE.fullmatch(translated):
                return []
            if any(character.isdigit() for character in translated):
                return []
            token = translated
    if SPAM_TOKEN_RE.search(token) or LAUGH_TOKEN_RE.fullmatch(token):
        return []
    token = ELONGATED_RE.sub(lambda match: match.group(1) * 2, token)
    normalized = REDUPLICATION_MAP.get(token, slang.get(token))
    if normalized is not None and SPAM_TOKEN_RE.search(normalized):
        return []
    if normalized is None:
        collapsed = re.sub(r"(?i)([a-z])\1+", r"\1", token)
        normalized = slang.get(collapsed, token)
    normalized = slang.get(normalized, normalized).strip()
    if not normalized:
        return []
    return RAW_TOKEN_RE.findall(normalized)


def get_stemmer():
    global _STEMMER
    if _STEMMER is None:
        _STEMMER = StemmerFactory().create_stemmer()
    return _STEMMER


def stem_token(token: str) -> str:
    if token in STEM_WHITELIST:
        return token
    cached = _STEM_CACHE.get(token)
    if cached is not None:
        return cached
    stemmed = get_stemmer().stem(token)
    if len(_STEM_CACHE) < 100_000:
        _STEM_CACHE[token] = stemmed
    return stemmed


def postprocess_final_tokens(tokens: list[str]) -> list[str]:
    cleaned: list[str] = []
    previous = None
    repeat_count = 0
    for token in tokens:
        if token == previous:
            repeat_count += 1
        else:
            previous = token
            repeat_count = 1
        max_repeat = 1 if token in SPECIAL_TOKENS else 2
        if repeat_count <= max_repeat:
            cleaned.append(token)
    return cleaned


def build_preprocessor(slang_broadcast):
    max_chars = env_int("PREPROCESSING_MAX_CHARS", MAX_PREPROCESS_CHARS)
    max_tokens = env_int("PREPROCESSING_MAX_TOKENS", MAX_PREPROCESS_TOKENS)
    use_stemming = env_bool("PREPROCESSING_USE_STEMMING", True)
    min_train_tokens = env_int("PREPROCESSING_MIN_TRAIN_TOKENS", MIN_TRAIN_TOKENS)

    def preprocess_text_base(text: Any, stemmer_func=None):
        raw = normalize_legal_terms(basic_clean_text(text))[:max_chars]
        raw_tokens = [token.casefold() for token in RAW_TOKEN_RE.findall(raw)]
        if is_spam_comment(raw):
            return (
                "",
                [],
                0,
                len(raw),
                len(raw_tokens),
                emoji.emoji_count(raw),
                len(URL_RE.findall(raw)),
                len(MENTION_RE.findall(raw)),
                len(HASHTAG_RE.findall(raw)),
                raw.count("!"),
                raw.count("?"),
                0.0,
                False,
                False,
                False,
                False,
                0,
                0,
            )
        letters = [character for character in raw if character.isalpha()]
        uppercase_count = sum(character.isupper() for character in letters)
        uppercase_ratio = uppercase_count / len(letters) if letters else 0.0

        feature_tokens: list[str] = []
        for token in raw_tokens:
            feature_tokens.extend(normalize_token(token, slang_broadcast.value))
        for hashtag in HASHTAG_RE.findall(raw):
            for token in RAW_TOKEN_RE.findall(expand_hashtag(hashtag)):
                feature_tokens.extend(normalize_token(token, slang_broadcast.value))

        working = URL_RE.sub(" url_token ", raw)
        working = MENTION_RE.sub(" user_mention ", working)
        working = HASHTAG_RE.sub(lambda match: f" {expand_hashtag(match.group(1))} ", working)
        working = replace_emojis(working).casefold()

        normalized_tokens: list[str] = []
        for token in RAW_TOKEN_RE.findall(working):
            normalized_tokens.extend(normalize_token(token, slang_broadcast.value))

        filtered_tokens = []
        for token in normalized_tokens:
            if token in CUSTOM_STOPWORDS:
                continue
            if token.isnumeric() and token not in NUMERIC_KEEP_TOKENS:
                continue
            if any(character.isdigit() for character in token) and token not in NUMERIC_KEEP_TOKENS and token not in STEM_WHITELIST:
                continue
            if len(token) == 1 and token not in NUMERIC_KEEP_TOKENS:
                continue
            if token in SPECIAL_TOKENS or token in PRESERVED_STOPWORDS:
                filtered_tokens.append(token)
            elif token not in STOPWORDS and not (token.isascii() and token in ENGLISH_STOPWORDS):
                filtered_tokens.append(token)
        filtered_tokens = filtered_tokens[:max_tokens]
        final_tokens = (
            [stemmer_func(token) for token in filtered_tokens]
            if stemmer_func is not None
            else filtered_tokens
        )
        final_tokens = [
            token
            for token in final_tokens
            if token
            and (not token.isnumeric() or token in NUMERIC_KEEP_TOKENS)
            and (
                token in SPECIAL_TOKENS
                or token in PRESERVED_STOPWORDS
                or token not in STOPWORDS
            )
        ]
        final_tokens = postprocess_final_tokens(final_tokens)

        domain_count = count_terms(feature_tokens, DOMAIN_TERMS)
        profanity_count = count_terms(feature_tokens, PROFANITY_TERMS)
        return (
            " ".join(final_tokens),
            final_tokens,
            len(filtered_tokens),
            len(raw),
            len(raw_tokens),
            emoji.emoji_count(raw),
            len(URL_RE.findall(raw)),
            len(MENTION_RE.findall(raw)),
            len(HASHTAG_RE.findall(raw)),
            raw.count("!"),
            raw.count("?"),
            float(round(uppercase_ratio, 6)),
            bool(letters and uppercase_ratio >= 0.9 and len(letters) >= 3),
            any(token in NEGATIONS for token in feature_tokens),
            domain_count > 0,
            profanity_count > 0,
            domain_count,
            profanity_count,
        )

    if use_stemming:
        def preprocess_text(text: Any):
            return preprocess_text_base(text, stem_token)

        return preprocess_text

    def preprocess_text(text: Any):
        return preprocess_text_base(text)

    return preprocess_text


def create_spark_session() -> SparkSession:
    mongo_uri = required_env("MONGO_URI")
    app_name = os.getenv("SPARK_APP_NAME", "IndonesianCommentStemmingPreprocessing").strip()
    master = os.getenv("SPARK_MASTER", "local[*]").strip()
    package = os.getenv("MONGO_SPARK_CONNECTOR_PACKAGE", MONGO_CONNECTOR_PACKAGE).strip()

    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
    os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)
    builder = (
        SparkSession.builder.appName(app_name)
        .master(master)
        .config("spark.jars.packages", package)
        .config("spark.mongodb.read.connection.uri", mongo_uri)
        .config("spark.mongodb.write.connection.uri", mongo_uri)
        .config("spark.sql.execution.pythonUDF.arrow.enabled", "false")
        .config("spark.sql.shuffle.partitions", os.getenv("SPARK_SQL_SHUFFLE_PARTITIONS", "4"))
        .config("spark.default.parallelism", os.getenv("SPARK_DEFAULT_PARALLELISM", "4"))
        .config("spark.driver.memory", os.getenv("SPARK_MEMORY", "1g"))
        .config("spark.pyspark.python", sys.executable)
        .config("spark.pyspark.driver.python", sys.executable)
    )
    if master.startswith("local"):
        os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
        builder = builder.config("spark.driver.host", "127.0.0.1").config(
            "spark.driver.bindAddress", "127.0.0.1"
        )
    return builder.getOrCreate()


def mongo_read(spark: SparkSession, database: str, collection: str) -> DataFrame:
    try:
        return (
            spark.read.format("mongodb")
            .option("database", database)
            .option("collection", collection)
            .option("aggregation.allowDiskUse", "true")
            .load()
        )
    except Exception as exc:
        raise RuntimeError(
            f"Gagal membaca MongoDB collection {database}.{collection}. "
            "Pastikan MongoDB aktif, URI benar, dan MongoDB Spark Connector tersedia."
        ) from exc


def mongo_write(df: DataFrame, database: str, collection: str, mode: str) -> None:
    try:
        (
            df.write.format("mongodb")
            .mode(mode)
            .option("database", database)
            .option("collection", collection)
            .save()
        )
    except Exception as exc:
        raise RuntimeError(
            f"Gagal menulis MongoDB collection {database}.{collection}: "
            f"{type(exc).__name__}: {exc}"
        ) from exc


def replace_collection_atomically(df: DataFrame, database: str, collection: str) -> None:
    mongo_uri = required_env("MONGO_URI")
    temp_collection = f"__tmp_{collection}_{uuid.uuid4().hex}"
    client = MongoClient(mongo_uri)
    db = client[database]
    try:
        db[temp_collection].drop()
        mongo_write(df, database, temp_collection, "overwrite")
        temp_count = db[temp_collection].count_documents({})
        if temp_count == 0:
            raise RuntimeError(f"Collection temporary {database}.{temp_collection} kosong; output final tidak diganti.")
        db[collection].drop()
        db[temp_collection].rename(collection, dropTarget=True)
    except Exception:
        db[temp_collection].drop()
        raise
    finally:
        client.close()


def count_pattern(df: DataFrame, pattern: str) -> int:
    return df.filter(F.col("text_original").rlike(pattern)).count()


def top_tokens(df: DataFrame, tokens_column: str, limit: int = 30) -> list[dict[str, Any]]:
    rows = (
        df.select(F.explode(F.col(tokens_column)).alias("token"))
        .filter(F.col("token") != "")
        .groupBy("token")
        .count()
        .orderBy(F.desc("count"), F.asc("token"))
        .limit(limit)
        .collect()
    )
    return [{"token": row["token"], "count": int(row["count"])} for row in rows]


def build_validation_report(input_df: DataFrame) -> dict[str, Any]:
    report: dict[str, Any] = {
        "total_rows": input_df.count(),
        "total_unique_videos": input_df.select("video_id").distinct().count()
        if "video_id" in input_df.columns
        else 0,
        "total_unique_authors": input_df.select("author").distinct().count()
        if "author" in input_df.columns
        else 0,
        "missing_text_original": input_df.filter(
            F.col("text_original").isNull() | (F.trim(F.col("text_original")) == "")
        ).count(),
        "duplicate_comment_id": 0,
        "duplicate_text_original": input_df.groupBy("text_original")
        .count()
        .filter(F.col("count") > 1)
        .agg(F.sum(F.col("count") - 1).alias("duplicates"))
        .first()["duplicates"]
        or 0,
        "short_comments": input_df.filter(F.length(F.trim(F.col("text_original"))) < 10).count(),
        "long_comments": input_df.filter(F.length(F.col("text_original")) > 1000).count(),
        "comments_with_emoji": count_pattern(input_df, r"[\x{1F000}-\x{1FAFF}]"),
        "comments_with_url": count_pattern(input_df, r"(?i)(https?://|www\.)"),
        "comments_with_mention": count_pattern(input_df, r"@\w+"),
        "comments_with_hashtag": count_pattern(input_df, r"#\w+"),
    }
    if "comment_id" in input_df.columns:
        report["duplicate_comment_id"] = (
            input_df.groupBy("comment_id")
            .count()
            .filter(F.col("comment_id").isNotNull() & (F.col("count") > 1))
            .agg(F.sum(F.col("count") - 1).alias("duplicates"))
            .first()["duplicates"]
            or 0
        )
    if "label" in input_df.columns:
        report["label_distribution"] = {
            str(row["label"]): int(row["count"])
            for row in input_df.groupBy("label").count().collect()
        }
    return report


def add_raw_tokens(df: DataFrame) -> DataFrame:
    cleaned = F.regexp_replace(
        F.lower(F.coalesce(F.col("text_original"), F.lit(""))),
        r"[^\p{L}\p{N}_]+",
        " ",
    )
    return df.withColumn(
        "_raw_report_tokens",
        F.filter(F.split(F.trim(cleaned), r"\s+"), lambda token: token != ""),
    )


def preprocess_dataframe(input_df: DataFrame, slang_broadcast) -> DataFrame:
    preprocess_udf = F.udf(build_preprocessor(slang_broadcast), PREPROCESS_SCHEMA)
    base_columns = [
        name
        for name in OUTPUT_INPUT_COLUMNS
        if name in input_df.columns and name not in GENERATED_COLUMNS
    ]
    base_df = input_df.select(*base_columns)
    processed = base_df.withColumn("_preprocessing", preprocess_udf(F.col("text_original")))
    processed = processed.select("*", "_preprocessing.*").drop("_preprocessing")
    duplicate_window = Window.partitionBy(FINAL_TEXT_COLUMN)
    duplicate_rank_window = Window.partitionBy(FINAL_TEXT_COLUMN).orderBy(F.col("comment_id"))
    return (
        processed.withColumn(
            "is_duplicate_text",
            (F.count(F.lit(1)).over(duplicate_window) > 1) & (F.col(FINAL_TEXT_COLUMN) != ""),
        )
        .withColumn("duplicate_text_rank", F.row_number().over(duplicate_rank_window))
        .withColumn("processed_at", F.current_timestamp())
    )


def filter_training_ready_dataframe(processed_df: DataFrame) -> DataFrame:
    min_train_tokens = env_int("PREPROCESSING_MIN_TRAIN_TOKENS", MIN_TRAIN_TOKENS)
    drop_empty = env_bool("PREPROCESSING_DROP_EMPTY_TEXT", True)
    drop_short = env_bool("PREPROCESSING_DROP_SHORT_TEXT", True)
    require_label = env_bool("PREPROCESSING_REQUIRE_LABEL", False) and "label" in processed_df.columns
    drop_duplicate_text = env_bool("PREPROCESSING_DROP_DUPLICATE_TEXT", True)

    condition = F.lit(True)
    if drop_empty:
        condition = condition & (F.col(FINAL_TEXT_COLUMN) != "")
    if drop_short:
        condition = condition & (F.col("normalized_token_count") >= min_train_tokens)
    if require_label:
        condition = condition & F.col("label").isNotNull() & (F.trim(F.col("label")) != "")
    if drop_duplicate_text and "duplicate_text_rank" in processed_df.columns:
        condition = condition & (F.col("duplicate_text_rank") == 1)
    return processed_df.filter(condition)


def select_output_dataframe(processed_df: DataFrame) -> DataFrame:
    output_columns = [
        name
        for name in [*OUTPUT_INPUT_COLUMNS, FINAL_TEXT_COLUMN]
        if name in processed_df.columns
    ]
    return processed_df.select(*output_columns)


def choose_output_collection(spark: SparkSession, database: str, requested: str) -> tuple[str, str]:
    return FIXED_OUTPUT_COLLECTION, "overwrite"


def build_final_report(
    validation: dict[str, Any],
    processed_df: DataFrame,
    output_ready_df: DataFrame,
    input_collection: str,
    output_collection: str,
    raw_top_tokens: list[dict[str, Any]],
) -> dict[str, Any]:
    totals = processed_df.agg(
        F.count("*").alias("total_rows_output"),
        F.sum(F.col("profanity_count")).alias("total_profanity"),
        F.sum(F.col("domain_term_count")).alias("total_domain_terms"),
        F.sum(F.when(F.col(FINAL_TEXT_COLUMN) == "", 1).otherwise(0)).alias("empty_text_final"),
        F.sum(F.when(F.col("normalized_token_count") < env_int("PREPROCESSING_MIN_TRAIN_TOKENS", MIN_TRAIN_TOKENS), 1).otherwise(0)).alias("too_short_text_final"),
        F.sum(F.when(F.col("is_duplicate_text"), 1).otherwise(0)).alias("duplicate_text_count"),
        F.sum(
            F.when(
                F.col("contains_negation")
                & ~F.col(FINAL_TEXT_COLUMN).rlike(
                    r"(^|\s)(tidak|bukan|jangan|belum|tanpa|kurang|tak)(\s|$)"
                ),
                1,
            ).otherwise(0)
        ).alias("lost_negation_count"),
        F.sum(
            F.when(
                F.col("contains_domain_terms")
                & ~F.col(FINAL_TEXT_COLUMN).rlike(
                    r"(^|\s)(tni|dpr|ruu|uu|sipil|militer|rakyat|negara|korupsi|koruptor|aset|"
                    r"perampasan|pemerintah|presiden|prabowo|polisi|demo|mahasiswa|orde|orba|"
                    r"oligarki|demokrasi|pasal)(\s|$)"
                ),
                1,
            ).otherwise(0)
        ).alias("lost_domain_term_count"),
    ).first()
    output_count = output_ready_df.count()
    warnings: list[str] = []
    if totals["empty_text_final"]:
        warnings.append(f"{totals['empty_text_final']} {FINAL_TEXT_COLUMN} kosong.")
    if totals["lost_negation_count"]:
        warnings.append(f"{totals['lost_negation_count']} komentar terindikasi kehilangan negasi.")
    if totals["too_short_text_final"]:
        warnings.append(f"{totals['too_short_text_final']} komentar kurang dari batas token training.")
    if totals["lost_domain_term_count"]:
        warnings.append(
            f"{totals['lost_domain_term_count']} komentar terindikasi kehilangan domain term."
        )
    if output_count == 0:
        warnings.append("Tidak ada baris yang lolos filter training-ready.")
    return {
        "run_id": str(uuid.uuid4()),
        "processed_at": datetime.now(timezone.utc).isoformat(),
        "input_collection": input_collection,
        "output_collection": output_collection,
        "total_rows_input": int(validation["total_rows"]),
        "total_rows_output": int(output_count),
        "total_unique_videos": int(validation["total_unique_videos"]),
        "total_unique_authors": int(validation["total_unique_authors"]),
        "missing_text_count": int(validation["missing_text_original"]),
        "duplicate_comment_id_count": int(validation["duplicate_comment_id"]),
        "duplicate_text_original_count": int(validation["duplicate_text_original"]),
        "duplicate_text_count": int(totals["duplicate_text_count"] or 0),
        "short_comments_count": int(validation["short_comments"]),
        "long_comments_count": int(validation["long_comments"]),
        "comments_with_emoji_count": int(validation["comments_with_emoji"]),
        "comments_with_url_count": int(validation["comments_with_url"]),
        "comments_with_mention_count": int(validation["comments_with_mention"]),
        "comments_with_hashtag_count": int(validation["comments_with_hashtag"]),
        "label_distribution_json": json.dumps(
            validation.get("label_distribution", {}), ensure_ascii=False
        ),
        "validation_metrics_json": json.dumps(validation, ensure_ascii=False),
        "top_tokens_before_json": json.dumps(raw_top_tokens, ensure_ascii=False),
        "top_tokens_after_json": json.dumps(top_tokens(output_ready_df, "tokens"), ensure_ascii=False),
        "total_profanity": int(totals["total_profanity"] or 0),
        "total_domain_terms": int(totals["total_domain_terms"] or 0),
        "empty_text_final_count": int(totals["empty_text_final"] or 0),
        "warnings": warnings,
    }


def main() -> None:
    load_dotenv(dotenv_path=".env", override=True)
    database = required_env("MONGO_DATABASE")
    input_collection = required_env("MONGO_INPUT_COLLECTION")
    requested_output_collection = os.getenv("MONGO_OUTPUT_COLLECTION", FIXED_OUTPUT_COLLECTION).strip() or FIXED_OUTPUT_COLLECTION
    report_collection = os.getenv("MONGO_REPORT_COLLECTION", FIXED_REPORT_COLLECTION).strip() or FIXED_REPORT_COLLECTION
    if requested_output_collection != FIXED_OUTPUT_COLLECTION:
        print(f"MONGO_OUTPUT_COLLECTION diabaikan: output fixed ke {FIXED_OUTPUT_COLLECTION}.", flush=True)
    if report_collection != FIXED_REPORT_COLLECTION:
        print(f"MONGO_REPORT_COLLECTION diabaikan: report fixed ke {FIXED_REPORT_COLLECTION}.", flush=True)
    requested_output_collection = FIXED_OUTPUT_COLLECTION
    report_collection = FIXED_REPORT_COLLECTION
    if requested_output_collection == input_collection:
        raise ValueError(
            "MONGO_OUTPUT_COLLECTION tidak boleh sama dengan MONGO_INPUT_COLLECTION."
        )

    spark = create_spark_session()
    spark.sparkContext.setLogLevel("WARN")
    print(
        f"Spark aktif: master={spark.sparkContext.master}; "
        f"input={database}.{input_collection}",
        flush=True,
    )
    try:
        input_df = mongo_read(spark, database, input_collection)
        total_rows = input_df.count()
        if total_rows == 0:
            raise ValueError(f"Collection input kosong: {database}.{input_collection}")
        if "text_original" not in input_df.columns:
            raise ValueError("Kolom text_original wajib tersedia pada collection input.")

        validation = build_validation_report(input_df)
        raw_report_df = add_raw_tokens(input_df)
        raw_top_tokens = top_tokens(raw_report_df, "_raw_report_tokens")
        slang_broadcast = spark.sparkContext.broadcast(SLANG_MAP)

        processed_df = preprocess_dataframe(input_df, slang_broadcast).persist(StorageLevel.DISK_ONLY)
        output_collection, write_mode = choose_output_collection(
            spark, database, requested_output_collection
        )
        output_ready_df = filter_training_ready_dataframe(processed_df).persist(StorageLevel.DISK_ONLY)
        output_df = select_output_dataframe(output_ready_df)
        replace_collection_atomically(output_df, database, output_collection)

        report = build_final_report(
            validation, processed_df, output_ready_df, input_collection, output_collection, raw_top_tokens
        )
        report_df = spark.createDataFrame([report], schema=REPORT_SCHEMA)
        mongo_write(report_df, database, report_collection, "append")

        print(json.dumps(report, ensure_ascii=False, indent=2, default=str), flush=True)
        print("\nContoh 10 hasil:", flush=True)
        preview_columns = ["text_original", FINAL_TEXT_COLUMN]
        if "label" in output_df.columns:
            preview_columns.append("label")
        output_df.select(*preview_columns).show(10, truncate=100)
    finally:
        spark.stop()



## Jalankan Pipeline

Membaca collection input, menyimpan hasil ke collection output, dan meng-append report ke collection report.

In [3]:
main()


Spark aktif: master=local[2]; input=analisis_sentimen.comments


d:\TugasUnud\Semester 6\BIG DATA\Analisis Sentimen\.venv\Lib\site-packages\pyspark\sql\udf.py:134: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


{
  "run_id": "210fd244-9821-47ba-9c39-a13a543a3ace",
  "processed_at": "2026-07-06T11:55:13.506154+00:00",
  "input_collection": "comments",
  "output_collection": "comments_preprocessed",
  "total_rows_input": 15663,
  "total_rows_output": 13177,
  "total_unique_videos": 5,
  "total_unique_authors": 13651,
  "missing_text_count": 11,
  "duplicate_comment_id_count": 0,
  "duplicate_text_original_count": 290,
  "duplicate_text_count": 1050,
  "short_comments_count": 517,
  "long_comments_count": 89,
  "comments_with_emoji_count": 3184,
  "comments_with_url_count": 16,
  "comments_with_mention_count": 382,
  "comments_with_hashtag_count": 72,
  "label_distribution_json": "{}",
  "validation_metrics_json": "{\"total_rows\": 15663, \"total_unique_videos\": 5, \"total_unique_authors\": 13651, \"missing_text_original\": 11, \"duplicate_comment_id\": 0, \"duplicate_text_original\": 290, \"short_comments\": 517, \"long_comments\": 89, \"comments_with_emoji\": 3184, \"comments_with_url\": 16, 

## Struktur Hasil

Notebook hanya menulis dua collection tujuan: `comments_preprocessed` untuk data training bersih dan `comments_preprocessing_report` untuk audit preprocessing. Collection `comments_preprocessed` dipangkas ke field penting saja: `comment_id`, `video_id`, `label` jika tersedia, `text_original`, dan `text_final`.
